# 65. Paper tables

Writes every table in the two Artemis II papers and their supplements to `tables/*.tex` from the
analysis outputs, so the papers `\input` them and a rerun refreshes them. Run after notebooks
20-60. Captions live here; edit them here, not in the `.tex` files.

| File | Used in | Source |
|---|---|---|
| `coupling.tex` | Paper 1, Table 1 | notebook 50 `coupling_metrics.csv` |
| `stations.tex` | Both supplements, Table S1 | `artemis2.yaml` station table + `INSTRUMENTS` below |
| `tote_modes.tex` | Paper 1 supplement, Table S2 | notebook 31 `tote_panel_modes.csv` |
| `amplitudes.tex` | Paper 2 supplement, Table S2 | notebook 40 `amplitude_field.csv` |

In [1]:
import numpy as np
import pandas as pd
import artemis2_config as cfg
from ksclib.latex import num, sig, signed, write_table

O = cfg.OUT_DIR
TAB = cfg.PAPER_DIR / "tables"

def short(seed_id):
    return seed_id.removeprefix("1R.")

## Table 1 (paper 1): air-to-ground coupling

In [2]:
c = pd.read_csv(O / "50_air_ground_coupling" / "coupling_metrics.csv").sort_values("distance_km", kind="stable")
t1 = pd.DataFrame({
    "Pair (pressure / velocity)": c.pair,
    "$r$ (km)": c.distance_km.map(lambda x: num(x, 2)),
    "$k_\\text{rms}$": c.k_rms_um_s_Pa.map(lambda x: num(x, 1)),
    "$k$": c.k_regression_um_s_Pa.map(lambda x: signed(x, 1)),
    "corr.": c["corr"].map(lambda x: signed(x, 2)),
    "$|H|_{1-5}$": c["H_1-5Hz_um_s_Pa"].map(sig),
    "$|H|_{5-20}$": c["H_5-20Hz_um_s_Pa"].map(sig),
    "$|H|_{20-40}$": c["H_20-40Hz_um_s_Pa"].map(sig),
    "$\\gamma^2_{5-20}$": c["coh_5-20Hz"].map(lambda x: num(x, 2)),
})
write_table(TAB / "coupling.tex", t1, label="tab:coupling", colfmt="lrrrrrrrr", size="scriptsize",
    caption=("Air-to-ground coupling at co-located sites, first 180~s after launch. $k_\\text{rms}$: RMS "
             "ratio of vertical velocity to pressure, 1--40~Hz. $k$: regression coefficient at the best lag "
             "($v = k\\,p$; negative means downward motion under compression; SmartSolo signs are as recorded, "
             "i.e.\\ reversed). $r$: correlation at that lag. $|H|$: median transfer-function magnitude where "
             "coherence $\\geq$~0.5 (--: none). $\\gamma^2$: median coherence, 5--20~Hz. Units "
             "\\si{\\micro\\metre\\per\\second\\per\\pascal}. B03's Centaur pair is faulty."))
t1

,Pair (pressure / velocity),$r$ (km),$k_\text{rms}$,$k$,corr.,$|H|_{1-5}$,$|H|_{5-20}$,$|H|_{20-40}$,$\gamma^2_{5-20}$
9,B14..HDF / B14..DPZ,0.74,4.5,+1.3,+0.28,0.72,2.2,8.2,0.71
0,B01.00.DD0 / B01.00.DHZ,1.53,7.5,$-$2.6,$-$0.35,0.58,5.1,7.1,0.30
4,B01.00.DD0 / B01..DPZ,1.53,7.5,+2.7,+0.35,--,5.3,7.1,0.32
5,B01..HDI / B01..DPZ,1.53,5.1,+1.4,+0.28,0.59,3.8,5.5,0.27
1,B03.00.DD0 / B03.00.DHZ,1.75,54.7,+5.0,+0.09,--,--,--,0.10
6,B03..HDI / B03..DPZ,1.75,6.9,$-$1.4,$-$0.21,1.3,2.8,3.5,0.16
8,B12..HDF / B12..DPZ,2.65,9.5,+3.2,+0.33,--,11,6.4,0.41
2,B05.00.DD0 / B05.00.DHZ,2.76,3.2,+0.8,+0.24,--,1.6,2.4,0.26
7,B07..HDF / B07..DPZ,3.42,6.0,+1.4,+0.24,--,2.0,1.5,0.36
14,B24..HDF / B24..DPZ,5.53,7.5,+2.1,+0.29,--,5.3,6.0,0.50


## Table S1 (both supplements): stations

Coordinates, distance and azimuth come from the station file (`artemis2.yaml`); the instruments at
each station are deployment metadata, listed here.

In [3]:
INSTRUMENTS = {
    "B14": "Trillium/Centaur, infraBSU, Gem, SmartSolo",
    "B01": "Trillium/Centaur, infraBSU, Gem, SmartSolo",
    "B03": "Trillium/Centaur, infraBSU, Gem, SmartSolo",
    "B12": "G\\\"{u}ralp Certimus, Gem, SmartSolo",
    "B05": "Trillium/Centaur, infraBSU",
    "B07": "G\\\"{u}ralp Certimus, Gem (2), SmartSolo",
    "B24": "Trillium/Centaur, infraBSU, Gem (2), SmartSolo",
    "B23": "Silicon Audio accel., Gem, SmartSolo",
    "B24RE": "Gem",
    "B29": "Trillium/Centaur, infraBSU, Gem (2), SmartSolo$^{a}$",
    "B24LT": "Gem",
    "B20W": "Gem, SmartSolo",
    "B20": "Trillium/Centaur, infraBSU, Chaparral M25, Gem (3), SmartSolo",
    "B20S": "Gem, SmartSolo",
}
st = cfg.station_table().sort_values("distance_km")
missing = set(st.index) ^ set(INSTRUMENTS)
assert not missing, f"station list and INSTRUMENTS differ: {missing}"
s1 = pd.DataFrame({
    "Station": st.index,
    "Lat ($^{\\circ}$)": st.lat.map(lambda x: num(x, 4)),
    "Lon ($^{\\circ}$)": st.lon.map(lambda x: num(x, 4)),
    "Distance (km)": st.distance_km.map(lambda x: num(x, 2)),
    "Azimuth ($^{\\circ}$)": st.azimuth_deg.map(lambda x: num(x, 1)),
    "Instrumentation": st.index.map(INSTRUMENTS),
})
write_table(TAB / "stations.tex", s1, label="tab:stations", colfmt="lrrrrp{5.2cm}", size="small",
    footnote="$^{a}$ No data during the launch.",
    caption=("Station locations and instrumentation, sorted by distance from SLC-39B. Gem 363, recorded under "
             "the provisional code \\texttt{1R.UNK..HD1}, was 8~m from B20 according to its GPS log."))
s1

,Station,Lat ($^{\circ}$),Lon ($^{\circ}$),Distance (km),Azimuth ($^{\circ}$),Instrumentation
station,,,,,,
B14,B14,28.6322,$-$80.6158,0.74,41.4,"Trillium/Centaur, infraBSU, Gem, SmartSolo"
B01,B01,28.6214,$-$80.6067,1.53,115.0,"Trillium/Centaur, infraBSU, Gem, SmartSolo"
B03,B03,28.6428,$-$80.6236,1.75,351.0,"Trillium/Centaur, infraBSU, Gem, SmartSolo"
B12,B12,28.6217,$-$80.6472,2.65,256.6,"G\""{u}ralp Certimus, Gem, SmartSolo"
B05,B05,28.6400,$-$80.6450,2.76,300.9,"Trillium/Centaur, infraBSU"
B07,B07,28.6072,$-$80.5942,3.42,130.4,"G\""{u}ralp Certimus, Gem (2), SmartSolo"
B24,B24,28.5806,$-$80.6408,5.53,200.7,"Trillium/Centaur, infraBSU, Gem (2), SmartSolo"
B23,B23,28.5786,$-$80.6078,5.54,166.7,"Silicon Audio accel., Gem, SmartSolo"
B24RE,B24RE,28.5812,$-$80.6433,5.56,203.3,Gem


## Table S2 (paper 1 supplement): tote panel bending modes

In [4]:
pm = pd.read_csv(O / "31_tote_acoustics" / "tote_panel_modes.csv")
pm["range"] = [f"{lo:.0f}--{hi:.0f}" for lo, hi in zip(pm.f_low_hz, pm.f_high_hz)]
pm["Panel"] = [f"{p} (${a:.2f} \\times {b:.3g}$~m)" for p, a, b in zip(pm.panel, pm.a_m, pm.b_m)]
s2 = pm.pivot(index="Panel", columns="mode", values="range")
s2 = s2.loc[pm.Panel.unique(), ["(1,1)", "(2,1)", "(1,2)"]].reset_index()
s2.columns.name = None
write_table(TAB / "tote_modes.tex", s2, label="tab:tote_modes", colfmt="lrrr", size="small",
            caption="Estimated in-vacuo bending-mode frequencies (Hz) of the tote panels.")
s2

,Panel,"(1,1)","(2,1)","(1,2)"
0,Base or lid ($0.66 \times 0.43$~m),8--19,15--36,24--58
1,Long wall ($0.66 \times 0.375$~m),10--23,17--40,31--75
2,Short wall ($0.43 \times 0.375$~m),13--31,29--70,34--83


## Table S2 (paper 2 supplement): peak amplitudes

In [5]:
a = pd.read_csv(O / "40_moving_source_amplitudes" / "amplitude_field.csv")
a = a[~((a.instrument == "infraBSU") & (a.station == "B03"))]        # hardware fault, omitted
NOTE = {"overflow: lower bound": "overflow", "faulty": "faulty"}
NOTE_OVERRIDE = {"1R.B05.00.DHZ": "drift"}                           # Trillium mass off-centre
note = a.flag.map(NOTE).where(a.flag.notna(), "")
note = [NOTE_OVERRIDE.get(s, n) for s, n in zip(a.seed_id, note)]
a = a.assign(note=note)

def block(d, pressure):
    d = d.sort_values("distance_km", kind="stable")
    if pressure:
        val = [("$\\geq$" if lb else "") + f"{p:.0f}" for p, lb in zip(d.peak, d.lower_bound)]
        lvl = (20 * np.log10(d.peak / 20e-6)).map(lambda x: f"{x:.0f}")
    else:
        val = [f"{v*1e3:.2f}" if v * 1e3 >= 0.1 else sig(v * 1e3) for v in d.peak]
        lvl = [""] * len(d)
    return pd.DataFrame({
        "Channel": d.seed_id.map(short), "$r$ (km)": d.distance_km.map(lambda x: num(x, 2)),
        "Az ($^{\\circ}$)": d.azimuth_deg.map(lambda x: f"{x:.0f}"), "Sensor": d.instrument,
        "$p_\\text{pk}$ (Pa)": val, "$L_\\text{pk}$ (dB)": list(lvl),
        "$t_\\text{pk}$ (s)": d.t_peak_s.map(lambda x: num(x, 1)), "Note": d.note})

p_block, v_block = block(a[a.kind == "pressure"], True), block(a[a.kind == "velocity"], False)
v_header = ("Channel & $r$ (km) & Az ($^{\\circ}$) & Sensor & $v_\\text{pk}$ (mm~s$^{-1}$) & & "
            "$t_\\text{pk}$ (s) & Note \\\\")
write_table(TAB / "amplitudes.tex", pd.concat([p_block, v_block], ignore_index=True),
    label="tab:amplitudes", colfmt="lrrlrrrl", size="scriptsize", placement="p",
    insert={len(p_block): ["\\midrule", v_header, "\\midrule"]},
    caption=("Peak pressure ($>$0.5~Hz) and peak vertical ground velocity ($>$1~Hz) in the first 180~s after "
             "launch, over unclipped samples. $\\geq$: overflowed Gem, value is the peak outside the overflow "
             "interval. $t_\\text{pk}$: time of the peak after launch. $L_\\text{pk}$: peak level, dB re "
             "\\SI{20}{\\micro\\pascal}. B03's infraBSU (DD0) is omitted (hardware fault)."))
pd.concat([p_block, v_block])

,Channel,$r$ (km),Az ($^{\circ}$),Sensor,$p_\text{pk}$ (Pa),$L_\text{pk}$ (dB),$t_\text{pk}$ (s),Note
0,B14..HDF,0.74,41,Gem,$\geq$95,134,31.5,overflow
1,B14..HDI,0.74,41,Gem,$\geq$90,133,31.5,overflow
2,B01..HDI,1.53,115,Gem,$\geq$99,134,35.6,overflow
3,B01.00.DD0,1.53,115,infraBSU,512,148,22.0,
4,B01.00.DD1,1.53,115,infraBSU,454,147,22.2,
5,B01.00.DD2,1.53,115,infraBSU,434,147,23.2,
6,B03..HDI,1.75,351,Gem,$\geq$128,136,5.6,overflow
7,B12..HDF,2.65,257,Gem,$\geq$98,134,38.0,overflow
8,B05.00.DD0,2.76,301,infraBSU,329,144,12.0,
9,B07..HDF,3.42,130,Gem,121,136,35.3,


In [6]:
print("Tables written to", TAB)
for p in sorted(TAB.glob("*.tex")):
    print(" ", p.name)

Tables written to /Users/glennthompson/Developer/KSCRocketSeismoHydrology/09_artemis2_paper/tables
  amplitudes.tex
  coupling.tex
  stations.tex
  tote_modes.tex
